# Deeper investigation — Schemas and parsing

**Learner exercise** · [All exercises](../../index.html) · [Setup](../../README.md)

Optional. Complete [Exercise 3](../03-validate.ipynb) and its **Save and finish** cell first. This investigation uses the same saved work; it does not replace your core pipeline.

Complete **Your code**, run the **Check** cells, and open hints when needed. Replace `todo(...)` with your answer. Do not use **Run All** while tasks remain unfinished.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Close the previous exercise after **Save and finish**. Missing earlier work? Use an explicit [catch-up step](../../RECOVERY.md).

In [ ]:
from pathlib import Path
import sys

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'workshop_runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

import lab_checks as check
from arrival_files import publish_arrival
from lab_checks import todo
from lab_workspace import Workspace
from workshop_runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path

workspace = Workspace(solutions=False)
product_key, clean_products, clean_sales, accepted_sales, rejected_sales = workspace.load('product_key', 'clean_products', 'clean_sales', 'accepted_sales', 'rejected_sales')
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
products = clean_products(raw_products)
cleaned = clean_sales(raw)
accepted = accepted_sales(cleaned)
rejected = rejected_sales(cleaned)
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

---
<a id="extension-schemas"></a>
## Your task

Read `data/extras/sales.csv` with its header and `raw.schema` into `csv_sales`. Compare its values and schema with Parquet. Would declaring `amount_raw` as a string reject `oops`?

Then read `data/extras/invalid_sales.csv` with that same schema into `extra_raw`. Apply your `clean_sales` and inspect the reasons. These extra rows are separate from the eight core inputs.

In [ ]:
csv_sales = todo("Read the CSV with header=True and raw.schema")
extra_raw = todo("Read the separate invalid-sales CSV with the same schema")
extra_cleaned = todo("Apply your cleaning function to the extra fixture")

In [ ]:
check.same_rows(raw, csv_sales)
check.extra_rejects(extra_cleaned)

<details><summary>Hint</summary>

Use `spark.read.option(...).schema(...).csv(...)`. A declared storage type and a business validation rule are different things.

</details>

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [ ]:
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Return to [all exercises](../../index.html).

After your attempt, compare the separate [worked solution](../../solutions/deeper/schemas-and-parsing.ipynb).